<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/5_2_Nonlinear_Nonparametric_Models_Bagging_and_Random_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part V. Nonlinear Nonparametric Models.
## 1. Decision Trees (CART)

## 2. Bagging and Random Forest

## 3. Gradient Boosting (XGBoost, LightGBM, CatBoost)


## 5.2. Бэггинг (Bootstrap Aggregating)

Одиночные решающие деревья страдают от высокой дисперсии: небольшие изменения в обучающей выборке способны радикально перестроить структуру дерева. Бэггинг (Bootstrap Aggregating), предложенный Брейманом (Breiman, 1996), уменьшает дисперсию предсказаний за счёт усреднения большого числа деревьев, обученных на случайно порождённых версиях исходных данных. Этот раздел посвящён математическим основаниям бэггинга, его свойствам и практической реализации.

### 1. Идея бэггинга

Пусть имеется обучающая выборка $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^n$. Бэггинг строит ансамбль из $B$ базовых моделей (как правило, решающих деревьев), каждая из которых обучается на **бутстреп‑выборке** $\mathcal{D}_b^*$, полученной из $\mathcal{D}$ случайным выбором $n$ объектов **с возвращением**. Таким образом, в каждой бутстреп‑выборке некоторые объекты исходной выборки могут присутствовать несколько раз, а некоторые — отсутствовать. Вероятность для конкретного объекта не попасть в $\mathcal{D}_b^*$ равна

$$
\left(1 - \frac{1}{n}\right)^n \xrightarrow{n\to\infty} e^{-1} \approx 0{,}368.
$$

Следовательно, каждая бутстреп‑выборка содержит в среднем около $63{,}2\%$ уникальных объектов исходной выборки.

Ансамблевое предсказание для регрессии строится как среднее арифметическое предсказаний отдельных моделей:

$$
\hat{f}_{\text{bag}}(\mathbf{x}) = \frac{1}{B} \sum_{b=1}^{B} \hat{f}_b(\mathbf{x}),
$$

где $\hat{f}_b$ — модель, обученная на $\mathcal{D}_b^*$. В задаче классификации агрегирование обычно проводят голосованием: большинством (hard voting) или усреднением предсказанных вероятностей классов (soft voting).

### 2. Математическое обоснование снижения дисперсии

Эффективность бэггинга основана на уменьшении дисперсии предсказаний за счёт усреднения. Рассмотрим идеализированную ситуацию: имеется $B$ случайных величин $\hat{f}_1(\mathbf{x}),\dots,\hat{f}_B(\mathbf{x})$, представляющих предсказания моделей ансамбля для фиксированной точки $\mathbf{x}$. Предположим, что все они имеют одинаковое математическое ожидание $\mathbb{E}[\hat{f}_b] = \mu$, одинаковую дисперсию $\mathrm{Var}(\hat{f}_b) = \sigma^2$ и попарную корреляцию $\rho = \mathrm{Corr}(\hat{f}_i, \hat{f}_j)$ для $i \neq j$. Тогда для среднего $\bar{f} = \frac{1}{B}\sum_{b=1}^B \hat{f}_b$ имеем:

$$
\begin{aligned}
\mathrm{Var}(\bar{f})
&= \frac{1}{B^2} \left( \sum_{i=1}^{B} \mathrm{Var}(\hat{f}_i) + \sum_{i \neq j} \mathrm{Cov}(\hat{f}_i, \hat{f}_j) \right) \\
&= \frac{1}{B^2} \bigl( B\sigma^2 + B(B-1)\rho\sigma^2 \bigr) \\
&= \frac{1}{B}\sigma^2 + \frac{B-1}{B}\rho\sigma^2 \\
&= \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2.
\end{aligned}
$$

При $\rho < 1$ дисперсия среднего строго меньше дисперсии одной модели: $\mathrm{Var}(\bar{f}) < \sigma^2$. С ростом $B$ второе слагаемое стремится к нулю, и дисперсия ансамбля асимптотически приближается к $\rho\sigma^2$. Таким образом, чем меньше корреляция между моделями, тем большее снижение дисперсии достигается; при $\rho = 0$ дисперсия уменьшается в $B$ раз.

#### Углублённый взгляд: асимптотическая дисперсия бэггинга и устойчивость алгоритма

Бутстреп‑ансамбль порождает модели, корреляция $\rho$ между которыми определяется **устойчивостью** базового алгоритма обучения. Для неустойчивых алгоритмов, таких как глубокие решающие деревья, малые изменения обучающей выборки (бутстреп) приводят к значительным вариациям предсказаний, поэтому корреляция между предсказаниями двух деревьев, обученных на разных бутстреп‑выборках, относительно невелика. Напротив, для устойчивых алгоритмов (например, $k$-ближайших соседей с большим $k$) бутстреп‑выборки почти не меняют модель, $\rho \approx 1$, и бэггинг не даёт выигрыша.

Формально, можно показать (см. Efron, Tibshirani, 1993; Buja, Stuetzle, 2006), что асимптотическая дисперсия бэггинг‑оценки связана с функцией влияния базового алгоритма. Для деревьев функция влияния неограничена, что и объясняет высокую эффективность бэггинга.

Бэггинг практически не изменяет смещение модели. Действительно, математическое ожидание предсказания ансамбля $\mathbb{E}[\bar{f}] = \frac{1}{B}\sum \mathbb{E}[\hat{f}_b] = \mu$ равно математическому ожиданию предсказания одной модели. Поскольку базовые модели (полные деревья) обладают низким смещением, ансамбль сохраняет это свойство, одновременно радикально снижая дисперсию.

### 3. Бэггинг для классификации

В задачах классификации с $K$ классами бэггинг‑ансамбль обычно использует один из двух механизмов агрегирования:

- **Жёсткое голосование (hard voting):** каждый классификатор отдаёт голос за один класс, итоговое решение — класс, набравший большинство голосов.
- **Мягкое голосование (soft voting):** каждый классификатор возвращает оценку вероятности $p_k^{(b)}(\mathbf{x})$ для каждого класса $k$; итоговая вероятность вычисляется как среднее $\bar{p}_k(\mathbf{x}) = \frac{1}{B}\sum_{b=1}^B p_k^{(b)}(\mathbf{x})$, после чего выбирается класс с максимальной вероятностью.

Мягкое голосование обычно предпочтительнее, так как учитывает «уверенность» каждой модели, что часто приводит к более высокому качеству.

Если классификаторы независимы и каждый ошибается с вероятностью $p < 1/2$, то вероятность ошибки ансамбля при жёстком голосовании даётся биномиальным хвостом:

$$
P_{\text{err}} = \sum_{k = \lfloor B/2 \rfloor + 1}^{B} \binom{B}{k} p^k (1-p)^{B-k}.
$$

Эта величина убывает экспоненциально с ростом $B$ (при $p < 1/2$), что теоретически обосновывает эффективность ансамблевого голосования. На практике корреляция между моделями несколько замедляет этот процесс, но общая тенденция сохраняется.

### 4. Out‑of‑Bag (OOB) оценка

Каждая бутстреп‑выборка $\mathcal{D}_b^*$ оставляет без использования около $36{,}8\%$ объектов исходной выборки. Эти объекты образуют **Out‑of‑Bag (OOB)** набор для $b$-й модели. Для каждого объекта $i$ можно получить OOB‑предсказание, усреднив (или проголосовав) только по тем моделям, в обучении которых этот объект не участвовал:

$$
\hat{y}_i^{\text{OOB}} = \frac{1}{|\mathcal{B}_i|} \sum_{b \in \mathcal{B}_i} \hat{f}_b(\mathbf{x}_i),
$$

где $\mathcal{B}_i = \{b \mid (\mathbf{x}_i, y_i) \notin \mathcal{D}_b^*\}$.

OOB‑ошибка, вычисленная на всех объектах, является практически несмещённой оценкой ошибки обобщения, сопоставимой по точности с кросс‑валидацией, но получаемой «бесплатно» в процессе обучения ансамбля. В отличие от обычной обучающей ошибки, OOB‑ошибка не страдает от оптимистического смещения, так как для каждого объекта используются только модели, не видевшие его при обучении.

### 5. Код и эксперименты

Проиллюстрируем свойства бэггинга на наборе данных `breast_cancer`. В качестве базовой модели возьмём полное решающее дерево (`max_depth=None`), которое характеризуется низким смещением и высокой дисперсией.




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score

# Загрузка данных
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

# Параметры эксперимента
B_range = [1, 3, 5, 10, 20, 50, 100, 200]
train_acc = []
test_acc = []
oob_acc = []

for B in B_range:
    # Bagging с полными деревьями, bootstrap=True, oob_score=True
    bag = BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=0),
        n_estimators=B,
        bootstrap=True,
        oob_score=True,
        random_state=42,
        n_jobs=-1
    )
    bag.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, bag.predict(X_train)))
    test_acc.append(accuracy_score(y_test, bag.predict(X_test)))
    oob_acc.append(bag.oob_score_)

# Визуализация
plt.figure(figsize=(8, 5))
plt.plot(B_range, train_acc, 'bo-', label='train')
plt.plot(B_range, test_acc, 'ro-', label='test')
plt.plot(B_range, oob_acc, 'g^--', label='OOB')
plt.xscale('log')
plt.xlabel('Число моделей B')
plt.ylabel('Accuracy')
plt.title('Бэггинг с полными решающими деревьями (breast_cancer)')
plt.legend()
plt.grid(True)
plt.show()


График демонстрирует характерное поведение: с увеличением числа моделей тестовая точность возрастает и насыщается, обучающая точность остаётся близкой к единице, а OOB‑оценка практически совпадает с тестовой. Это подтверждает, что бэггинг эффективно уменьшает дисперсию, не увеличивая смещение, и что OOB‑ошибка служит надёжным индикатором обобщающей способности.



### Вопросы для самопроверки

#### Для начинающих
1. Зачем нужен бэггинг и какую проблему одиночных решающих деревьев он решает?
2. Что такое бутстреп‑выборка и почему в ней примерно $63\%$ уникальных объектов?
3. Почему OOB‑ошибка считается почти несмещённой оценкой качества обобщения?
4. Как бэггинг влияет на смещение и дисперсию базовой модели?

#### Для профессионалов
1. Докажите формулу $\mathrm{Var}(\bar{f}) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$ для среднего коррелированных случайных величин с равными дисперсиями и попарной корреляцией $\rho$.
2. Объясните, почему бэггинг не уменьшает смещение. Приведите математическое обоснование с использованием линейности математического ожидания.
3. Сравните OOB‑ошибку и кросс‑валидацию с точки зрения смещения и дисперсии. В каких случаях OOB‑ошибка может давать смещённую оценку?
4. Выведите вероятность ошибки ансамбля при жёстком голосовании для $B$ независимых классификаторов, каждый из которых ошибается с вероятностью $p$, и покажите, что она экспоненциально убывает с ростом $B$ при $p < 1/2$.


## 5.2. Случайный лес: декорреляция деревьев, важность признаков и теория

Бэггинг над решающими деревьями снижает дисперсию, усредняя предсказания многих моделей. Однако деревья, построенные по разным бутстреп‑выборкам, всё ещё могут быть сильно скоррелированы, если в данных присутствует несколько доминирующих признаков: каждое дерево будет использовать их в первых же расщеплениях, что приводит к схожей структуре и, как следствие, к высокой попарной корреляции предсказаний. Случайный лес (Random Forest), предложенный Брейманом (Breiman, 2001), решает эту проблему, вводя дополнительный источник случайности — при каждом расщеплении выбор признака ограничивается случайным подмножеством размера $m$ из полного набора $D$ признаков. Эта простая модификация радикально повышает точность ансамбля.

### 1. От бэггинга к случайному лесу

В случайном лесе сохраняется бутстреп‑агрегирование базовых деревьев. Главное отличие заключено в процедуре построения каждого дерева:

> Для каждого узла **перед расщеплением** из полного набора $D$ признаков случайным образом (без возвращения) отбирается подмножество размера $m$, и оптимальное расщепление ищется только среди этого подмножества.

Типичные рекомендации для выбора $m$, подтверждённые обширными экспериментами:

- **Классификация:** $m = \big\lfloor \sqrt{D} \big\rfloor$,
- **Регрессия:** $m = \big\lfloor D/3 \big\rfloor$.

Деревья строятся до максимальной глубины (без прунинга), часто вплоть до листьев, содержащих минимальное число объектов. Это сохраняет низкое смещение каждого отдельного дерева. Ансамблевое предсказание, как и в бэггинге, формируется усреднением (регрессия) или голосованием (классификация).

Математическая мотивация случайного выбора признаков непосредственно следует из формулы дисперсии среднего коррелированных случайных величин, выведенной ранее:

$$
\mathrm{Var}\big(\bar{f}\big) = \rho\,\sigma^2 + \frac{1-\rho}{B}\,\sigma^2.
$$

Если несколько признаков являются очень сильными, то бэггинг‑деревья будут похожи друг на друга ($\rho$ велико), и дисперсия ансамбля снижается незначительно. Ограничение числа кандидатов $m$ **принудительно** заставляет деревья использовать различные признаки, что уменьшает среднюю попарную корреляцию $\rho$. Пусть корреляция между предсказаниями двух деревьев, обученных на бутстреп‑выборках, в бэггинге равна $\rho_{\text{bag}}$, а в случайном лесе $\rho_{\text{RF}}$. Тогда при $m < D$ выполняется $\rho_{\text{RF}} < \rho_{\text{bag}}$, а значит, и дисперсия ансамбля $\mathrm{Var}_{\text{RF}} < \mathrm{Var}_{\text{bag}}$. При $m = D$ случайный лес вырождается в бэггинг.

Вместе с тем, чрезмерное уменьшение $m$ увеличивает смещение каждого отдельного дерева, потому что важные признаки могут быть исключены из рассмотрения во многих узлах, что ухудшает качество подгонки. Оптимальное значение $m$ находится как компромисс между декорреляцией (снижение дисперсии ансамбля) и сохранением точности отдельных деревьев (контроль смещения).

### 2. Анализ смещения и дисперсии для случайного леса

Проведём более детальный анализ компромисса «смещение–дисперсия» для случайного леса. Обозначим через $\hat{f}_b(\mathbf{x})$ предсказание $b$-го дерева, а через $\bar{f}(\mathbf{x}) = \frac{1}{B}\sum_{b=1}^B \hat{f}_b(\mathbf{x})$ — предсказание ансамбля. Математическое ожидание ансамбля равно

$$
\mathbb{E}\big[\bar{f}(\mathbf{x})\big] = \frac{1}{B}\sum_{b=1}^B \mathbb{E}\big[\hat{f}_b(\mathbf{x})\big] = \mu(\mathbf{x}),
$$

поскольку все деревья строятся по одному и тому же вероятностному механизму. Таким образом, ансамбль практически не меняет смещение по сравнению с одним деревом.

Дисперсия каждого отдельного дерева зависит от $m$: чем меньше $m$, тем меньше информации используется при расщеплении, и тем выше дисперсия предсказаний одного дерева (как функции обучающей выборки). Однако усреднение по ансамблю преобразует индивидуальную дисперсию $\sigma^2$ и попарную корреляцию $\rho$ в общую дисперсию согласно формуле $\rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$. Уменьшение $m$ снижает $\rho$ (положительный эффект) и одновременно может увеличить $\sigma^2$ (отрицательный эффект). При достаточно большом числе деревьев $B$ определяющим становится первое слагаемое $\rho\sigma^2$, поэтому снижение корреляции является приоритетной целью, даже ценой умеренного роста индивидуальной дисперсии. Этим объясняется, почему стандартные эвристики $m = \sqrt{D}$ или $m = D/3$ оказываются близкими к оптимальным для широкого круга задач.

### 3. Out‑of‑Bag (OOB) оценка в случайном лесе

Каждое дерево случайного леса обучается на своей бутстреп‑выборке, содержащей в среднем около $63{,}2\%$ уникальных объектов исходной выборки. Оставшиеся объекты, не попавшие в бутстреп‑выборку, образуют **Out‑of‑Bag** (OOB) набор для этого дерева. Для каждого объекта $i$ можно построить OOB‑предсказание, усреднив (или проголосовав) только по тем деревьям, в обучении которых этот объект не участвовал:

$$
\hat{y}_i^{\text{OOB}} = \frac{1}{|\mathcal{B}_i|} \sum_{b \in \mathcal{B}_i} \hat{f}_b(\mathbf{x}_i),
$$

где $\mathcal{B}_i = \{b \mid (\mathbf{x}_i, y_i) \notin \mathcal{D}_b^*\}$.

OOB‑ошибка, вычисленная на всех объектах, является практически несмещённой оценкой ошибки обобщения случайного леса. В отличие от обучающей ошибки, OOB‑ошибка не страдает от оптимистического смещения, так как для каждого объекта используются только модели, не видевшие его при обучении. OOB‑оценка получается «бесплатно» в процессе построения леса и часто используется для настройки гиперпараметров без выделения валидационной выборки.

### 4. Важность признаков в случайном лесе

Одно из главных достоинств случайного леса — возможность оценить вклад каждого признака в предсказание. Вводится две основные меры важности: основанная на уменьшении impurity (MDI) и основанная на перестановке (permutation importance).

#### 4.1. Уменьшение impurity (Mean Decrease in Impurity, MDI)

Для каждого признака $j$ суммируется уменьшение критерия impurity (например, Gini или энтропия) по всем узлам всех деревьев, в которых признак $j$ использовался для расщепления. Уменьшение взвешивается числом обучающих объектов, прошедших через узел. Формально, для одного дерева $T$:

$$
\text{FI}_j(T) = \sum_{t \in T: \text{split}(t) = j} \frac{m_t}{n} \,\Delta I_t,
$$

где $m_t$ — число объектов в узле $t$, а

$$
\Delta I_t = I_t - \frac{m_{t,L}}{m_t} I_{t,L} - \frac{m_{t,R}}{m_t} I_{t,R}
$$

— уменьшение impurity в узле $t$. Итоговая MDI‑важность получается усреднением $\text{FI}_j(T)$ по всем деревьям леса.

MDI вычислительно дёшева, так как все величины $\Delta I_t$ и $m_t$ известны из процесса обучения. Однако MDI обладает известным смещением в пользу признаков с большим числом уникальных значений или категорий. Такие признаки предоставляют больше возможных разбиений, и даже если они не связаны с целевой переменной, жадный алгоритм может выбрать одно из них, создавая ложное уменьшение impurity. Кроме того, для признаков, имеющих более широкий диапазон значений, выше вероятность того, что случайно выбранный порог даст заметное уменьшение impurity.

#### 4.2. Permutation Importance (важность по перестановке)

Идея, предложенная Брейманом (2001), состоит в измерении падения качества предсказаний при разрушении связи между признаком и целевой переменной. Для каждого признака $j$ его значения случайным образом перемешиваются по наблюдениям (пермутация), после чего вычисляется ошибка модели на изменённых данных. Важность определяется как средняя разность между ошибкой на перемешанных данных и исходной ошибкой:

$$
\text{FI}_j = \frac{1}{R}\sum_{r=1}^R \bigl[ \text{Err}(\tilde{X}_j^{(r)}) - \text{Err}_{\text{orig}} \bigr],
$$

где $\tilde{X}_j^{(r)}$ — матрица признаков с $r$-й случайной перестановкой $j$-го столбца, а $R$ — число повторных перестановок.

Ошибка обычно измеряется как разность в accuracy (для классификации) или в MSE (для регрессии). В качестве исходной ошибки $\text{Err}_{\text{orig}}$ берётся ошибка модели на исходных данных (обычно OOB‑ошибка или ошибка на отложенной выборке).

Главное достоинство permutation importance — независимость от структуры модели и отсутствие смещения в сторону категорийных признаков. Однако для сильно коррелированных признаков перестановка одного из них не всегда полностью разрушает информацию (дублирующий признак может частично компенсировать), что иногда приводит к заниженным оценкам важности каждого из коррелированной группы. Современные модификации, такие как **Conditional Permutation Importance**, переставляют признак условно на другие признаки, что позволяет более честно оценивать вклад в присутствии коллинеарности.

#### 4.3. Сравнение MDI и Permutation Importance

Две меры часто коррелируют, но могут давать различный порядок значимости. MDI склонна завышать важность многозначных и категориальных признаков, тогда как permutation importance не обладает этим недостатком. На практике рекомендуется анализировать обе меры, а при противоречиях — больше доверять permutation importance или использовать её условную версию.

#### Углублённый взгляд: связь MDI с объяснённой дисперсией в регрессии

В регрессионном случайном лесе, использующем MSE в качестве impurity, суммарное уменьшение MSE по всем узлам леса обладает прозрачной вероятностной интерпретацией. Для одного дерева сумма $\sum_t \frac{m_t}{n} \Delta \text{MSE}_t$ равна разности между дисперсией отклика в корневом узле и средневзвешенной дисперсией в листьях — то есть доле дисперсии, «объяснённой» разбиениями. Усредняя по лесу, получаем, что MDI признака $j$ асимптотически (с ростом числа деревьев) оценивает вклад этого признака в объяснённую дисперсию. Более формально, рассматривая случайный лес как адаптивную оценку функции регрессии, MDI можно связать с функциональным разложением ANOVA и оценками чувствительности (см. работы Штробля и др., 2007; Грёмпе, 2009). Это делает MDI особенно интерпретируемой мерой для регрессии. Для классификации интерпретация через уменьшение энтропии менее прямая, но сохраняет аналогичный смысл «уменьшения неопределённости» целевой переменной.

### 5. Эксперименты

Продемонстрируем свойства случайного леса на наборе данных `breast_cancer`. Обучим лес из 500 деревьев с рекомендованным параметром $m = \lfloor\sqrt{D}\rfloor$, сравним с бэггингом и одиночным деревом, исследуем важность признаков и влияние $m$ на качество. Также воспользуемся встроенной OOB‑оценкой.



In [ ]:
# @title Случайный лес: обучение, OOB, важность признаков и влияние m
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score

# Загрузка данных
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

D = X.shape[1]
m_default = int(np.sqrt(D))
print(f"Число признаков: {D}, рекомендованное m: {m_default}")

# Обучение моделей (для леса включаем OOB-подсчёт)
rf = RandomForestClassifier(n_estimators=500, max_features=m_default,
                            random_state=42, n_jobs=-1, oob_score=True)
rf.fit(X_train, y_train)

bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=0),
    n_estimators=500, random_state=42, n_jobs=-1)
bag.fit(X_train, y_train)

tree = DecisionTreeClassifier(random_state=0)
tree.fit(X_train, y_train)

# Сравнение точности и OOB
models = {'Одиночное дерево': tree, 'Бэггинг (500)': bag,
          'Случайный лес (500)': rf}
for name, model in models.items():
    acc = accuracy_score(y_test, model.predict(X_test))
    oob_str = f", OOB = {model.oob_score_:.4f}" if hasattr(model, 'oob_score_') else ""
    print(f"{name}: test accuracy = {acc:.4f}{oob_str}")


Ожидается, что случайный лес превзойдёт и одиночное дерево, и бэггинг благодаря дополнительной декорреляции, а OOB‑оценка будет близка к тестовой точности.

#### Важность признаков (MDI и Permutation)

Извлечём MDI из обученного леса и рассчитаем permutation importance на тестовой выборке. Для сравнения построим горизонтальные диаграммы топ‑10 признаков и scatter‑plot двух мер.




In [ ]:
# MDI (встроенная важность)
mdi_importances = rf.feature_importances_
indices_mdi = np.argsort(mdi_importances)[::-1]

# Permutation importance на тесте
perm_result = permutation_importance(rf, X_test, y_test, n_repeats=10,
                                     random_state=42, n_jobs=-1)
perm_importances = perm_result.importances_mean
indices_perm = np.argsort(perm_importances)[::-1]

# Визуализация топ-10 признаков по каждой мере
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(range(10), mdi_importances[indices_mdi][:10][::-1])
axes[0].set_yticks(range(10))
axes[0].set_yticklabels([data.feature_names[i] for i in indices_mdi[:10][::-1]])
axes[0].set_title('MDI importance (топ-10)')
axes[0].set_xlabel('Уменьшение impurity')

axes[1].barh(range(10), perm_importances[indices_perm][:10][::-1])
axes[1].set_yticks(range(10))
axes[1].set_yticklabels([data.feature_names[i] for i in indices_perm[:10][::-1]])
axes[1].set_title('Permutation importance (топ-10)')
axes[1].set_xlabel('Падение accuracy')

plt.tight_layout()
plt.show()

# Сравнение двух мер (scatter plot всех признаков)
plt.figure(figsize=(7,6))
plt.scatter(mdi_importances, perm_importances)
plt.xlabel('MDI importance')
plt.ylabel('Permutation importance')
plt.title('MDI vs Permutation importance')
for i in range(D):
    if mdi_importances[i] > 0.05 or perm_importances[i] > 0.02:
        plt.annotate(data.feature_names[i], (mdi_importances[i], perm_importances[i]),
                     fontsize=8, alpha=0.7)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


Scatter‑plot позволяет заметить признаки, для которых MDI велика, а permutation importance — мала (потенциальное смещение MDI).

#### Зависимость точности от $m$

Зафиксируем $B=500$ и проварьируем `max_features` от 1 до $D$. Для каждого $m$ зафиксируем также OOB‑ошибку.




In [ ]:
m_values = np.arange(1, D+1)
test_acc = []
oob_acc = []
for m in m_values:
    rf_temp = RandomForestClassifier(n_estimators=500, max_features=m,
                                     random_state=42, n_jobs=-1, oob_score=True)
    rf_temp.fit(X_train, y_train)
    test_acc.append(accuracy_score(y_test, rf_temp.predict(X_test)))
    oob_acc.append(rf_temp.oob_score_)

plt.figure(figsize=(8,5))
plt.plot(m_values, test_acc, 'o-', label='Test accuracy')
plt.plot(m_values, oob_acc, 's-', label='OOB accuracy')
plt.axvline(m_default, color='r', linestyle='--', label=f'default m={m_default}')
plt.xlabel('m (max_features)')
plt.ylabel('Accuracy')
plt.title('Влияние числа случайных признаков на точность случайного леса')
plt.legend()
plt.grid()
plt.show()


График демонстрирует, что точность растёт с увеличением $m$, достигает плато в районе рекомендованного значения, после чего может незначительно снижаться. OOB‑оценка практически повторяет поведение тестовой, подтверждая свою пригодность для настройки гиперпараметров без выделения валидационной выборки.

#### Стабилизация ошибки с ростом числа деревьев

В отличие от многих других алгоритмов, случайный лес **не переобучается** при увеличении числа деревьев $B$: тестовая ошибка монотонно убывает и стабилизируется около некоторого асимптотического уровня. Проиллюстрируем это, зафиксировав $m = m_{\text{default}}$ и варьируя $B$.




In [ ]:
B_values = np.logspace(1, 3, 15, dtype=int)  # от 10 до 1000 с логарифмическим шагом
test_err = []
oob_err = []
for B in B_values:
    rf_temp = RandomForestClassifier(n_estimators=B, max_features=m_default,
                                     random_state=42, n_jobs=-1, oob_score=True)
    rf_temp.fit(X_train, y_train)
    test_err.append(1 - accuracy_score(y_test, rf_temp.predict(X_test)))
    oob_err.append(1 - rf_temp.oob_score_)

plt.figure(figsize=(8,5))
plt.plot(B_values, test_err, 'o-', label='Test error')
plt.plot(B_values, oob_err, 's-', label='OOB error')
plt.xscale('log')
plt.xlabel('Число деревьев B')
plt.ylabel('Ошибка')
plt.title('Сходимость ошибки случайного леса с ростом B')
plt.legend()
plt.grid()
plt.show()


Как видно, ошибка стабилизируется уже при нескольких десятках деревьев; дальнейшее увеличение $B$ лишь незначительно улучшает результат, но не приводит к переобучению.


### 6. Числовая иллюстрация построения случайного леса

Для закрепления теории рассмотрим сквозной числовой пример, демонстрирующий ключевые этапы работы случайного леса: формирование бутстреп‑выборки, построение одного дерева со случайным подмножеством признаков, вычисление impurity, выбор порога, агрегирование предсказаний и оценку важности признаков.

**Данные.**  
Пусть обучающая выборка содержит $n = 6$ объектов с двумя числовыми признаками $x_1$ и $x_2$ и бинарной целевой переменной $y \in \{0, 1\}$:

| $i$ | $x_1$ | $x_2$ | $y$ |
|-----|-------|-------|-----|
| 1   | 2     | 3     | 0   |
| 2   | 1     | 2     | 0   |
| 3   | 3     | 1     | 0   |
| 4   | 5     | 4     | 1   |
| 5   | 6     | 5     | 1   |
| 6   | 4     | 6     | 1   |

Это линейно разделимые данные: класс $0$ сосредоточен в левом нижнем углу, класс $1$ — в правом верхнем.

**Параметры леса (для иллюстрации).**  
Число деревьев $B = 2$, число случайно отбираемых признаков в каждом узле $m = 1$ (при $D = 2$ по формуле $\lfloor\sqrt{2}\rfloor = 1$). Деревья строятся до глубины $2$ без ограничения на минимальное число объектов в листе.

#### 6.1. Первое дерево

**Бутстреп‑выборка.**  
Из исходной выборки объёмом $n = 6$ случайным образом (с возвращением) сформируем бутстреп‑выборку. Допустим, выпали индексы $\{1, 2, 2, 4, 5, 6\}$. Тогда данные для первого дерева:

| $i$ (в бутстрепе) | $x_1$ | $x_2$ | $y$ |
|-------------------|-------|-------|-----|
| 1 (объект 1)      | 2     | 3     | 0   |
| 2 (объект 2)      | 1     | 2     | 0   |
| 3 (объект 2)      | 1     | 2     | 0   |
| 4 (объект 4)      | 5     | 4     | 1   |
| 5 (объект 5)      | 6     | 5     | 1   |
| 6 (объект 6)      | 4     | 6     | 1   |

Объект 3 исходной выборки не попал в бутстреп, а объект 2 дублирован.

**Корневой узел.**  
В корневом узле находятся все 6 объектов. Оценим распределение классов: три объекта класса $0$, три объекта класса $1$. Impurity по индексу Джини:

$$
I_{\text{root}} = 1 - \left(\frac{3}{6}\right)^2 - \left(\frac{3}{6}\right)^2 = 1 - \frac{1}{4} - \frac{1}{4} = 0.5.
$$

Согласно $m = 1$, случайно отбираем один признак из двух. Пусть выбран $x_1$. Ищем оптимальный порог $\theta$ для разбиения. Уникальные значения $x_1$ в узле: $\{1, 2, 4, 5, 6\}$ (с учётом дубликата $1$). Пороги-кандидаты — середины между соседними значениями: $\{1.5, 3.0, 4.5, 5.5\}$.

Для каждого порога вычисляем взвешенную impurity дочерних узлов:

- $\theta = 1.5$: левый узел — объекты с $x_1 \le 1.5$ (три объекта: все $y=0$), правый узел — остальные (три объекта: все $y=1$). Тогда $I_{\text{left}} = 0$, $I_{\text{right}} = 0$, взвешенная impurity $= 0$. Уменьшение impurity: $\Delta I = 0.5 - 0 = 0.5$.

- $\theta = 3.0$: левый узел — три объекта класса $0$ (все), правый — три объекта класса $1$ (все). Аналогично $\Delta I = 0.5$.

- $\theta = 4.5$: левый узел — три объекта класса $0$ и один объект класса $1$ (объект 4 с $x_1=5$ попадает в правый). Тогда в левом узле: класс $0$ — $3$, класс $1$ — $0$? Проверим: при $\theta = 4.5$ объекты с $x_1 \le 4.5$ — это объекты 1,2,2,4? У объекта 4 $x_1=5 > 4.5$, значит он в правом. Левый: объекты 1,2,2 — все $y=0$, impurity $0$. Правый: объекты 4,5,6 — все $y=1$, impurity $0$. И снова $\Delta I = 0.5$.

- $\theta = 5.5$: левый узел — объекты 1,2,2,4,6? Нет, у объекта 6 $x_1=4 \le 5.5$, он в левом. Тогда левый: четыре объекта (три класса $0$, один класса $1$ — объект 4 с $y=1$). Правый: объект 5 ($y=1$). Тогда $I_{\text{left}} = 1 - (3/4)^2 - (1/4)^2 = 1 - 9/16 - 1/16 = 6/16 = 0.375$, $I_{\text{right}} = 0$ (чистый). Взвешенная impurity $= \frac{4}{6} \cdot 0.375 + \frac{2}{6} \cdot 0 = 0.25$. $\Delta I = 0.5 - 0.25 = 0.25$.

Максимальное $\Delta I = 0.5$ достигается при порогах $1.5, 3.0, 4.5$. Выберем, например, $\theta = 3.0$. Тогда левое поддерево ($x_1 \le 3$) содержит объекты $\{1,2,2\}$ (все $y=0$), правое ($x_1 > 3$) — объекты $\{4,5,6\}$ (все $y=1$). Оба дочерних узла чисты, impurity равна нулю, дальнейшие разбиения не требуются — они становятся листьями с предсказаниями $0$ и $1$ соответственно. Глубина дерева достигла $1$, что меньше разрешённой $2$, но узлы уже чисты.

Итак, первое дерево: корень — проверка $x_1 \le 3.0$, левый лист — класс $0$, правый лист — класс $1$. Признак $x_2$ в этом дереве не использовался.

#### 6.2. Второе дерево

Сгенерируем другую бутстреп‑выборку. Пусть выпали индексы $\{3, 3, 4, 4, 5, 6\}$ (объекты 3 и 4 повторены, объекты 1 и 2 отсутствуют). Данные:

| $i$ (в бутстрепе) | $x_1$ | $x_2$ | $y$ |
|-------------------|-------|-------|-----|
| 1 (объект 3)      | 3     | 1     | 0   |
| 2 (объект 3)      | 3     | 1     | 0   |
| 3 (объект 4)      | 5     | 4     | 1   |
| 4 (объект 4)      | 5     | 4     | 1   |
| 5 (объект 5)      | 6     | 5     | 1   |
| 6 (объект 6)      | 4     | 6     | 1   |

Корневой узел: два объекта класса $0$, четыре объекта класса $1$. Impurity:

$$
I_{\text{root}} = 1 - \left(\frac{2}{6}\right)^2 - \left(\frac{4}{6}\right)^2 = 1 - \frac{4}{36} - \frac{16}{36} = \frac{16}{36} \approx 0.444.
$$

Снова $m = 1$, случайно выбираем один признак. Допустим, выпал $x_2$. Уникальные значения $x_2$: $\{1, 4, 5, 6\}$. Пороги-кандидаты: $\{2.5, 4.5, 5.5\}$.

- $\theta = 2.5$: левый узел — объекты с $x_2 \le 2.5$ (два объекта класса $0$), правый — четыре объекта класса $1$. Оба чисты. $\Delta I = 0.444 - 0 = 0.444$.
- $\theta = 4.5$: левый — два объекта класса $0$ и два объекта класса $1$ (объекты 3,3 с $y=0$ и 4,4 с $y=1$). Тогда $I_{\text{left}} = 1 - (2/4)^2 - (2/4)^2 = 0.5$. Правый — два объекта класса $1$ (чист, impurity $0$). Взвешенная impurity $= \frac{4}{6} \cdot 0.5 + \frac{2}{6} \cdot 0 \approx 0.333$. $\Delta I = 0.444 - 0.333 = 0.111$.
- $\theta = 5.5$: левый — два объекта класса $0$ и четыре объекта класса $1$? Проверим: при $\theta = 5.5$ объекты с $x_2 \le 5.5$ — это объекты 3,3,4,4,5 (объект 6 с $x_2=6$ не входит). Тогда левый: два класса $0$, три класса $1$. $I_{\text{left}} = 1 - (2/5)^2 - (3/5)^2 = 1 - 4/25 - 9/25 = 12/25 = 0.48$. Правый: один объект класса $1$ (чист). Взвешенная impurity $= \frac{5}{6} \cdot 0.48 + \frac{1}{6} \cdot 0 = 0.4$. $\Delta I = 0.444 - 0.4 = 0.044$.

Максимальное $\Delta I$ даёт порог $2.5$, создающий два чистых дочерних узла. Выбираем его. Левое поддерево ($x_2 \le 2.5$) — объекты $\{3,3\}$ (класс $0$), правое ($x_2 > 2.5$) — объекты $\{4,4,5,6\}$ (класс $1$). Оба листа чисты.

Второе дерево: корень проверяет $x_2 \le 2.5$, левый лист — класс $0$, правый — класс $1$. Признак $x_1$ не использовался.

#### 6.3. Агрегирование и предсказание

Теперь у нас есть ансамбль из двух деревьев. Как классифицировать новый объект, например, $\mathbf{x}_* = (3.5, 3.5)$?

- Первое дерево: проверяет $x_1 \le 3.0$. У объекта $x_1 = 3.5 > 3.0$, поэтому он попадает в правый лист, предсказание класса $1$.
- Второе дерево: проверяет $x_2 \le 2.5$. У объекта $x_2 = 3.5 > 2.5$, поэтому правое поддерево, предсказание класса $1$.

Оба дерева голосуют за класс $1$. Итоговое предсказание — класс $1$.

Рассмотрим другой объект $\mathbf{x}_* = (2.0, 1.5)$:

- Первое дерево: $x_1 = 2.0 \le 3.0$, левый лист, предсказание $0$.
- Второе дерево: $x_2 = 1.5 \le 2.5$, левый лист, предсказание $0$.

Итог — класс $0$.

Объект $\mathbf{x}_* = (4.0, 2.0)$:

- Первое дерево: $x_1 = 4.0 > 3.0$, предсказание $1$.
- Второе дерево: $x_2 = 2.0 \le 2.5$, левый лист, предсказание $0$.

Здесь голоса разделились: одно дерево за класс $1$, другое за класс $0$. При жёстком голосовании с нечётным числом деревьев был бы выбран класс большинства; при двух деревьях можно предусмотреть правило случайного выбора или опираться на мягкое голосование (усреднение вероятностей). В реальных лесах с сотнями деревьев подобные коллизии исчезают.

#### 6.4. Важность признаков

Оценим MDI и Permutation Importance на нашем ансамбле из двух деревьев, используя исходную обучающую выборку (а не бутстреп‑выборки).

**MDI (Mean Decrease in Impurity).**  
Вспомним, что MDI признака $j$ равна сумме взвешенных уменьшений impurity по всем узлам всех деревьев, где признак использовался для разбиения. Для первого дерева: признак $x_1$ использовался в корневом узле с $m_t = 6$, $\Delta I = 0.5$. Вклад в MDI: $\frac{6}{6} \cdot 0.5 = 0.5$ (нормировка на исходный размер выборки $n=6$). Для второго дерева: признак $x_2$ в корне с $m_t = 6$, $\Delta I \approx 0.444$, вклад $0.444$. Итоговые MDI (усреднённые по двум деревьям): $x_1 \mapsto 0.25$, $x_2 \mapsto 0.222$. (При большем числе деревьев значения усредняются более полно.)

**Permutation Importance.**  
Оценим важность, измеряя падение точности на исходных данных после случайной перестановки значений признака. Исходная точность ансамбля на обучающей выборке (все 6 объектов) составляет $100\%$, так как каждое дерево безошибочно классифицирует свои бутстреп‑выборки, а вместе они правильно предсказывают все объекты (проверьте: объект 3 не был в первом дереве, но второе дерево его классифицирует верно). Перемешаем случайно значения признака $x_1$ между объектами. Допустим, после перемешивания столбец $x_1$ стал $(4, 5, 2, 1, 6, 3)$ (случайная перестановка). Прогоним через оба дерева и зафиксируем ошибку. Повторим перестановку несколько раз и усредним разность между полученной ошибкой и исходной. Аналогично для $x_2$. Поскольку в нашем маленьком примере деревья идеально разделяют классы, любая перестановка, скорее всего, приведёт к ошибке. Точные числа можно получить, выполнив несколько итераций вручную или в коде, но принципиальный результат: оба признака получат положительную важность, причём более важный признак (тот, который чаще используется в корневых разбиениях) будет иметь большее значение.

Этот пример наглядно демонстрирует, как случайный лес комбинирует деревья, использующие разные признаки, и как вычисляются меры важности. В реальных приложениях с сотнями деревьев и признаков описанные шаги полностью автоматизированы, но понимание их механики позволяет осознанно интерпретировать результаты и настраивать модель.


### 6. Итог и рекомендации

Случайный лес — один из наиболее надёжных и универсальных алгоритмов машинного обучения. Его основные достоинства:

- автоматическое снижение дисперсии без увеличения смещения;
- встроенная оценка обобщающей способности через OOB;
- информативные меры важности признаков (MDI и permutation);
- устойчивость к выбросам, шуму и пропущенным данным;
- минимальная настройка гиперпараметров.

При практическом использовании рекомендуется:

- Начинать со стандартных значений $m = \sqrt{D}$ (классификация) или $m = D/3$ (регрессия).
- Число деревьев $B$ брать порядка нескольких сотен; контролировать сходимость можно по OOB‑ошибке.
- Анализировать важность признаков, применяя как MDI, так и permutation importance; при наличии сильно коррелированных признаков отдавать предпочтение условной permutation importance.
- Сравнивать случайный лес с градиентным бустингом, который при правильной настройке может давать более высокую точность, но более чувствителен к гиперпараметрам и склонен к переобучению.


### Вопросы для самопроверки

#### Для начинающих
1. Чем случайный лес отличается от обычного бэггинга над решающими деревьями?
2. Зачем при построении деревьев в случайном лесе используется случайное подмножество признаков?
3. Как измерить важность признака в случайном лесе? Опишите идею MDI и Permutation Importance.
4. Что лучше: много глубоких деревьев или много неглубоких деревьев в случайном лесе, и почему?
5. Что такое OOB‑оценка и почему она полезна?

#### Для профессионалов
1. Выведите формулу дисперсии ансамбля для среднего коррелированных предсказаний и покажите, как выбор $m$ влияет на корреляцию $\rho$ и, следовательно, на итоговую дисперсию.
2. Объясните причину смещения MDI в пользу признаков с большим числом категорий. Предложите способ коррекции этого смещения.
3. Сравните permutation importance и SHAP (SHapley Additive exPlanations) с точки зрения учёта корреляций между признаками и вычислительной сложности.
4. Как теоретически выбор $m$ влияет на смещение и дисперсию каждого отдельного дерева и всего ансамбля? Опишите компромисс «смещение–дисперсия» в контексте случайного леса.
5. Докажите, что с увеличением числа деревьев $B$ ошибка случайного леса сходится к пределу, который не зависит от $B$, и объясните, почему переобучения не происходит.